# Get to Know a Dataset: YAOKI Lunar Rover IM-2 Mission Telemetry and Imagery Dataset

This notebook serves as a guided tour of the [YAOKI Lunar Rover](https://registry.opendata.aws/im2-yaoki-rover) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

### Organization of the YAOKI Rover dataset.

At the top level of our S3 bucket, three files describe the project:

 1. The dataset license (LICENSE.txt)
 2. A PDF document explaining the mission and high level data structure (PublicRelease_Overview_Lunar_Ledger_YAOKI_Dymon_JAOPS.pdf)
 3. A PDF document showing a more in depth data analysis (PublicRelease_YAOKI_IM2_data_analysis_by_JAOPS.pdf)

And three directories that contain the project data:

 1. A directory /images which contains the image data acquired by the rover. `.png` files are reconstructed images for easy viewing.  `.raw` files are a rover-specific custom format containing the raw image telemetry.  Packet layout and decoding are explained in detail in the [yaoki_images.ipynb](https://github.com/jaops-space/im2-yaoki-yamcs-public/blob/main/analysis/yaoki_images.ipynb) notebook.
 2. A directory /timeseries which contains the telemetry and telecommand data from the rover in parquet format.  Each stream has one parquet and a yaml describing the parquet columns, datatypes and units.  `manifest.yaml` lists all the data streams.  Full data analysis of telemetry and telecommand (tmtc) is in [yaoki_tmtc.ipynb](https://github.com/jaops-space/im2-yaoki-yamcs-public/blob/main/analysis/yaoki_tmtc.ipynb) notebook.
 3. A directory /yamcs-data which contains the telemetry and telecommand data from the rover as a Yamcs archive in RocksDB format.  Reading the data requires running a Yamcs server, but one can reproduce the full mission control environment.  More details are given in the [repository README](https://github.com/jaops-space/im2-yaoki-yamcs-public#option-2-docker-setup).

<img src="https://raw.githubusercontent.com/jaops-space/im2-yaoki-yamcs-public/main/s3-bucket-structure.svg"
     alt="Diagram showing the structure of data in the S3 bucket."
     width="100%">

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# botocore >= 1.38.23
# matplotlib >= 3.10.3
# numpy >= 1.26.0
# pandas >= 2.2.0
# PyYAML >= 6.0
# IPython >= 8.20.0
# plotly >= 5.24.0
# pyarrow >= 15.0.0

import os
import botocore
import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import yaml
from datetime import datetime, timezone
from IPython.display import Code
import plotly.express as px

In [ ]:
# Connect to the S3 server
BUCKET = "im2-yaoki-rover-public"
REGION = "ap-northeast-1"

s3 = boto3.client("s3", region_name=REGION, config=botocore.config.Config(signature_version=botocore.UNSIGNED))

### Download and display one processed image from the YAOKI rover.

The actual process of converting the data packets sent from the rover into an image is explained in full detail in the notebook: [yaoki_images.ipynb](https://github.com/jaops-space/im2-yaoki-yamcs-public/blob/main/analysis/yaoki_images.ipynb).

The purpose here is to show only a single reconstructed frame.

In [ ]:
# Create download directory for the images data for YAOKI.
os.makedirs('images', exist_ok=True)

# Download one image from Amazon S3.
IMAGE_NAME = 'images/imager_00000_00029.png'
s3.download_file(BUCKET, IMAGE_NAME, IMAGE_NAME)

# Read the image from disk
img = plt.imread(IMAGE_NAME)

# Convert it to grayscale.
gray = 0.2126*img[:,:,0] + 0.7152*img[:,:,1] + 0.0722*img[:,:,2]

# Plot only the pixels with between 1-99% intensity on the histogram 
vmin = np.percentile(img, 1)
vmax = np.percentile(img, 99)
plt.imshow(gray, cmap='gray', vmin=vmin, vmax=vmax)
_ = plt.colorbar()
# Arrow points to detached rover part.
_ = plt.annotate("", xy=(295, 105), xytext=(320, 85),
             arrowprops=dict(arrowstyle="->", color="red", lw=2))

### Answering questions about the landing using the images.

The IM2 lander did not settle on its feet and the YAOKI rover's position at the bottom of the lander was uniquely beneficial for visualizing what happened during the landing.  The red arrow above points to a footpad sticker that detached during landing.  For more information and discussion view p.23 of the [mission outline pdf](https://github.com/jaops-space/im2-yaoki-yamcs-public/blob/main/PublicRelease_YAOKI_IM2_data_analysis_by_JAOPS.pdf).

### YAOKI rover telemetry information.

Exploration of all the telemetry and telecommand streams is given in detail in the notebook: [yaoki_tmtc.ipynb](https://github.com/jaops-space/im2-yaoki-yamcs-public/blob/main/analysis/yaoki_tmtc.ipynb).

The purpose here is to show only how to locate and visualize one stream.  We will ultimately plot the temperature sensor that was sitting on the CPU of the main board as explained on p.12 of the [mission outline pdf](https://github.com/jaops-space/im2-yaoki-yamcs-public/blob/main/PublicRelease_YAOKI_IM2_data_analysis_by_JAOPS.pdf).  The first step is to download and read the manifest.yaml file which lists all the telemetry and telecommand streams which are archived, as well as basic metadata about them.

In [ ]:
# Download the manifest showing all the telemetry and telecommand streams.
PARQUET_DIR = Path('timeseries')
os.makedirs(PARQUET_DIR, exist_ok=True)
FILE_NAME = PARQUET_DIR / 'manifest.yaml'
s3.download_file(BUCKET, str(FILE_NAME), str(FILE_NAME))

# Read the manifest and list all the telemetry streams in alphabetical order
MANIFEST = yaml.safe_load(Path(FILE_NAME).read_text())

# Alphabetize and print out all the telemetry streams.
for k in sorted(MANIFEST["telemetry_streams"]):
    # If this is, in particular the ADC_TEMP4 stream we are interested in,
    # then print the whole yaml block underneath.
    if k == "/YAOKI/Rover/ADC_TEMP4":
        print(yaml.dump(
            {k: MANIFEST["telemetry_streams"][k]},
            sort_keys=False
        ).rstrip())
    else:
        print(k)

#### We found the stream we are interested in (ADC_TEMP04).

Next, we need to download and plot the information for that stream.  There are two files the comprise the stream.  The parquet file contains the actual data.  The yaml file contains the metadata describing the fields.

In [ ]:
# Now we need to download the files for this timestream.
# Make a download directory for them.
os.makedirs(PARQUET_DIR / "telemetry", exist_ok=True)

ADC_TEMP4_PARQUET = MANIFEST["telemetry_streams"]["/YAOKI/Rover/ADC_TEMP4"]["file"] # Build the filename
print(f"Downloading {ADC_TEMP4_PARQUET}")
s3.download_file(BUCKET, str(PARQUET_DIR / ADC_TEMP4_PARQUET), str(PARQUET_DIR / ADC_TEMP4_PARQUET))

ADC_TEMP4_YAML = MANIFEST["telemetry_streams"]["/YAOKI/Rover/ADC_TEMP4"]["metadata"]
print(f"Downloading {ADC_TEMP4_YAML}")
s3.download_file(BUCKET, str(PARQUET_DIR / ADC_TEMP4_YAML), str(PARQUET_DIR / ADC_TEMP4_YAML))


In [ ]:
# Let's inspect the metadata yaml for the ADC_TEMP4 stream now. 
METADATA = yaml.safe_load((PARQUET_DIR / ADC_TEMP4_YAML).read_text())
Code(yaml.dump(METADATA, sort_keys=False), language="yaml")

#### Now plot the temperature sensor timestream data.

Note that the temperature sensors do not report values outside their calibrated range and therefore values below -40 °C are not reported.

In [ ]:
# The temperature sensors only function down to a temperature of -40 Celsius.
TEMPERATURE_LIMIT_CELSIUS = -40
YAOKI_FIRST_TELEMETRY_DATE = datetime.fromisoformat("2025-03-07 02:17:15.150000+00:00")
YAOKI_LAST_TELEMETRY_DATE = datetime.fromisoformat("2025-03-07 04:33:04.389000+00:00")

In [ ]:
# Read the temperature sensor data from parquet  
df_ADC_TEMP4 = pd.read_parquet(PARQUET_DIR / ADC_TEMP4_PARQUET)

# Make a nice plot!
fig = px.scatter(
    df_ADC_TEMP4,
    x="generation_time",
    y="eng_value",
    title="ADC_TEMP4 Temperature Sensor readings",
    labels={"value": "Temperature (°C)", "timestamp": "Time (UTC)"},
)
fig.add_hline(
    y=TEMPERATURE_LIMIT_CELSIUS,
    line_color="black",
    annotation_text="rover temperature sensor limit",
    annotation_position="bottom",
)
fig.update_xaxes(title_text="Measurement Time")
fig.update_yaxes(title_text="Temperature (°C)")
fig.add_vline(x=YAOKI_FIRST_TELEMETRY_DATE.astimezone(timezone.utc), line_dash="dash", line_color="black")
fig.add_vline(x=YAOKI_LAST_TELEMETRY_DATE.astimezone(timezone.utc), line_dash="dash", line_color="black")
fig.update_layout(yaxis_range=[-65, 0])
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.4))

fig.show()

### What can we learn from this dataset?

There are potentially a number of data that could be teased out of this dataset given time and interest.  Some ideas that we have not had a chance to explore (yet):

- Is there any relationship between rover temperature and image noise in the camera images?
- Some data packets were not received during image transmissions.  Is this correlated with radio signal (RSSI)?
- Simulate total data transmission rate with parity options, varying color encodings, etc. and determine if a better protocol could have been used.
- The temperature as a function of time can be interpolated/extrapolated by combining the lander and rover values (see [yaoki_tmtc.ipynb](https://github.com/jaops-space/im2-yaoki-yamcs-public/blob/main/analysis/yaoki_tmtc.ipynb)).  Is there a better model for this which could be used on future missions?
- Lander temperature during flight is recorded.
- The accelerometer shows some "noisy" readings.  Do these match timings when motors or electronics ran?  This could be used, for example, to verify whether the motors or electronic systems were functioning correctly, or the presence of "noise" correlation with one motor command but lack of another could indicate as command issue.